# Homework 10: KV Cache, In-Context Learning, and Supervised Fine-Tuning

---

### Objectives

In this assignment you will practice three key concepts from Lecture 10:

1. **KV Cache** -- Implement the Key-Value cache mechanism that makes autoregressive LLM inference efficient.
2. **In-Context Learning & Chain-of-Thought** -- Compare instruction-following and reasoning capabilities between a large model (GPT-4o) and a smaller instruct model using the Azure OpenAI API.
3. **Supervised Fine-Tuning (SFT)** -- Implement a toy SFT pipeline that transforms a base language model into an instruction-following model.

---

### Skills You Will Practice

- Understanding how KV Cache avoids redundant computation during token generation
- Computing attention scores using cached keys and values
- Calling LLM APIs and designing prompts for in-context learning
- Comparing model capabilities across scales
- Implementing the SFT data format and training loop

---

### Guidelines

- **Do not modify** the structure or logic of the provided code -- your task is to understand and complete it.
- Problem 1 runs on CPU (no API needed). Problems 2-3 require the Azure OpenAI endpoint from your Foundry account.
- Total: **25 points**

---

### Setup

In [1]:
import subprocess, sys

def install_if_missing(package):
    try:
        __import__(package)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "--quiet"])

install_if_missing("torch")
install_if_missing("numpy")
install_if_missing("openai")

import warnings
warnings.filterwarnings("ignore")

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

torch.manual_seed(42)

## Problem 1: KV Cache Simulation (40 points)

### Background

During autoregressive generation, an LLM produces tokens one at a time. Without KV Cache, the model must recompute the Key and Value projections for ALL previous tokens at every step. With KV Cache, we store previously computed K and V vectors and only compute new ones for the current token.

### Recap from lecture

- **Without cache:** At step t, compute Q, K, V for all t tokens (O(t^2) total work across all steps).
- **With cache:** At step t, compute Q, K, V only for the new token. Retrieve cached K, V for previous tokens.
- **Prefill phase:** Process the entire prompt at once, populate the cache.
- **Decode phase:** Generate one token at a time, appending to the cache.

In this problem, you will simulate KV Cache step-by-step using the same numerical example from lecture: predicting "I love to learn...".

In [10]:
# Weight matrices for Q, K, V projections (from lecture)
# Input dimension = 3, output dimension = 2

W_Q = torch.tensor([[0.1, 0.2],
                    [0.3, 0.1],
                    [0.2, 0.4]])

W_K = torch.tensor([[0.4, 0.1],
                    [0.2, 0.3],
                    [0.1, 0.2]])

W_V = torch.tensor([[0.5, 0.1],
                    [0.2, 0.2],
                    [0.3, 0.4]])

# Token embeddings (fake vectors for illustration)
x_I    = torch.tensor([[0.2, 0.3, 0.4]])   # "I"
x_love = torch.tensor([[0.1, 0.5, 0.2]])   # "love"
x_to   = torch.tensor([[0.2, 0.2, 0.1]])   # "to"

print("Weight matrices loaded.")
print(f"W_Q shape: {list(W_Q.shape)}")
print(f"Input tokens: 'I' -> {x_I.tolist()}, 'love' -> {x_love.tolist()}, 'to' -> {x_to.tolist()}")

Weight matrices loaded.
W_Q shape: [3, 2]
Input tokens: 'I' -> [[0.20000000298023224, 0.30000001192092896, 0.4000000059604645]], 'love' -> [[0.10000000149011612, 0.5, 0.20000000298023224]], 'to' -> [[0.20000000298023224, 0.20000000298023224, 0.10000000149011612]]


In [11]:
# TODO Step 1: Process "I" -- Build the first cache entry (3 points)
#
# Hints:
# - Compute q1 = x_I @ W_Q (matrix multiply the input with the query weight matrix)
# - Compute k1 = x_I @ W_K (key projection)
# - Compute v1 = x_I @ W_V (value projection)
# - Initialize kv_cache_keys = [k1] and kv_cache_values = [v1]
# - Since there's only one token, the attention weight is just 1.0
# - Context vector = 1.0 * v1

# TODO: Compute projections
q1 = x_I @ W_Q
k1 = x_I @ W_K
v1 = x_I @ W_V

print("Step 1: Processing 'I'")
print(f"  q1 = {q1}")
print(f"  k1 = {k1}")
print(f"  v1 = {v1}")

# TODO: Initialize the KV cache
kv_cache_keys = [k1]    # Should contain k1
kv_cache_values = [v1]  # Should contain v1

# TODO: Compute attention (trivial: only one token)
context1 = v1  # Should be v1 (weight = 1.0)
print(f"  Context vector = {context1}")
print(f"  Cache size: {len(kv_cache_keys)}")

Step 1: Processing 'I'
  q1 = tensor([[0.1900, 0.2300]])
  k1 = tensor([[0.1800, 0.1900]])
  v1 = tensor([[0.2800, 0.2400]])
  Context vector = tensor([[0.2800, 0.2400]])
  Cache size: 1


In [12]:
# TODO Step 2: Process "love" -- Use cached k1, v1 (4 points)
#
# Hints:
# - Compute q2, k2, v2 for x_love (same projection matrices)
# - Append k2, v2 to the cache
# - Compute attention scores: q2 dot each cached key
#   score_I = (q2 * kv_cache_keys[0]).sum().item()
#   score_love = (q2 * kv_cache_keys[1]).sum().item()
# - Apply softmax to get weights: F.softmax(torch.tensor([score_I, score_love]), dim=0)
# - Context = weighted sum of cached values

# TODO: Compute new projections for "love"
q2 = x_love @ W_Q
k2 = x_love @ W_K
v2 = x_love @ W_V

# TODO: Append to cache
kv_cache_keys.append(k2)
kv_cache_values.append(v2)

# TODO: Compute attention scores against ALL cached keys
score_I = (q2 * kv_cache_keys[0]).sum().item()
score_love = (q2 * kv_cache_keys[1]).sum().item()

# TODO: Softmax
weights = F.softmax(torch.tensor([score_I, score_love]), dim=0)

# TODO: Compute context vector (weighted sum of cached values)
context2 = weights[0] * kv_cache_values[0] + weights[1] * kv_cache_values[1]

print("Step 2: Processing 'love'")
print(f"  Weights: {weights.tolist()}")
print(f"  Context vector = {context2.tolist()}")
print(f"  Cache size: {len(kv_cache_keys)}")

Step 2: Processing 'love'
  Weights: [0.5006250143051147, 0.49937501549720764]
  Context vector = [[0.24504375457763672, 0.21503126621246338]]
  Cache size: 2


In [13]:
# TODO Step 3: Process "to" -- Use cached k1, v1, k2, v2 (3 points)
#
# Hints:
# - Same pattern as Step 2, but now q3 attends to 3 cached keys
# - Compute q3, k3, v3 for x_to
# - Append to cache
# - Score against ALL 3 cached keys
# - Softmax over 3 scores
# - Context = weighted sum of 3 cached values

# TODO: Compute projections for "to"
q3 = x_to @ W_Q
k3 = x_to @ W_K
v3 = x_to @ W_V

# TODO: Append to cache
kv_cache_keys.append(k3)
kv_cache_values.append(v3)

# TODO: Compute scores against all 3 cached keys
score_I = (q3 * kv_cache_keys[0]).sum().item()
score_love = (q3 * kv_cache_keys[1]).sum().item()

# TODO: Softmax over 3 scores
weights = F.softmax(torch.tensor([score_I, score_love]), dim=0)

# TODO: Compute context (weighted sum of all 3 cached values)
context3 = weights[0] * kv_cache_values[0] + weights[1] * kv_cache_values[1]

print("Step 3: Processing 'to'")
print(f"  Context vector = {context3.tolist()}")
print(f"  Cache size: {len(kv_cache_keys)}")

print("\n--- Key Insight ---")
print("At step 3, we only computed q3, k3, v3 (the NEW token).")
print("We REUSED k1, v1, k2, v2 from the cache -- no recomputation!")

Step 3: Processing 'to'
  Context vector = [[0.24501749873161316, 0.21501249074935913]]
  Cache size: 3

--- Key Insight ---
At step 3, we only computed q3, k3, v3 (the NEW token).
We REUSED k1, v1, k2, v2 from the cache -- no recomputation!


## Problem 2: In-Context Learning and Chain-of-Thought (40 points)

### Background

GPT-3 demonstrated that large language models can perform tasks from examples in the prompt alone (few-shot in-context learning) without any gradient updates. Chain-of-Thought (CoT) prompting further improves reasoning by asking the model to show intermediate steps.

In this problem, you will compare **GPT-4o** (a large, highly-aligned model) with **Phi-4-mini-instruct** (a smaller instruct model) on instruction-following and reasoning tasks.

### Recap from lecture

- **Zero-shot:** No examples, just the instruction.
- **Few-shot:** Provide examples in the prompt, model learns the pattern on the fly.
- **Chain-of-Thought:** Add "Let's think step by step" or show reasoning examples.
- Larger models follow instructions more precisely and reason more reliably.

In [3]:
# Azure OpenAI Client Setup (use your endpoint from Foundry)
# You should have already set this up in HW9.

! pip install openai azure-identity --quiet

from azure.identity import DeviceCodeCredential, get_bearer_token_provider
from openai import AzureOpenAI

credential = DeviceCodeCredential()
token_provider = get_bearer_token_provider(
    credential,
    "https://cognitiveservices.azure.com/.default"
)

client = AzureOpenAI(
    azure_endpoint="https://mlearn530nusmith-aiservices.services.ai.azure.com/",
    azure_ad_token_provider=token_provider,
    api_version="2024-02-01"
)

print("Client ready. Models available: gpt-4o, Phi-4-mini-instruct")

Client ready. Models available: gpt-4o, Phi-4-mini-instruct


In [22]:
# TODO Part A: Compare instruction following (5 points)
#
# You will send the same prompts to both GPT-4o and Phi-4-mini-instruct
# and observe how well each model follows strict instructions.
#
# Hints:
# - Use client.chat.completions.create(model=model_name, messages=[...], temperature=0.0)
# - messages should include both a "system" message and a "user" message
# - Loop through models_to_test for each prompt
# - Print the response: response.choices[0].message.content

models_to_test = ["gpt-4o", "Phi-4-mini-instruct"]

test_prompts = [
    {
        "name": "Strict JSON Output",
        "system": "You are a strict data scientist. Output ONLY valid JSON. No other text. Structure: {\"summary\": [\"point 1\", \"point 2\"]}.",
        "user": "Summarize the history of AI in two bullet points."
    },
    {
        "name": "One-Word Constraint",
        "system": "You are a helpful assistant. You must follow constraints exactly.",
        "user": "There are 5 apples in a room. I ate 2. How many are left? Answer in exactly one word."
    },
    {
        "name": "Role Adherence",
        "system": "You are a pirate. Every response must be in pirate speak. Never break character.",
        "user": "What is machine learning?"
    }
]

# TODO: Loop through test_prompts, and for each prompt loop through models_to_test
# Send the prompt and print the response from each model
for i, prompt in enumerate(test_prompts):
    print(f"\nTEST {i+1}: {prompt['name']}")
    print(f"User: {prompt['user']}")

    for model_name in models_to_test:
        try:
            response = client.chat.completions.create(
                model=model_name,
                messages=[
                    {"role": "system", "content": prompt["system"]},
                    {"role": "user", "content": prompt["user"]}
                ]
            )

            print(f"\nModel: {model_name}")
            print(response.choices[0].message.content)

        except Exception as e:
            print(f"\nModel: {model_name}")
            print(f"ERROR: {e}")


TEST 1: Strict JSON Output
User: Summarize the history of AI in two bullet points.
To sign in, use a web browser to open the page https://login.microsoft.com/device and enter the code ARJBFE4VF to authenticate.

Model: gpt-4o
{"summary": ["AI development began in the mid-20th century, focusing on symbolic reasoning and basic problem-solving.", "The field evolved with advancements in machine learning, neural networks, and deep learning, leading to modern applications across industries."]}

Model: Phi-4-mini-instruct
- AI, or artificial intelligence, has a history that dates back to the mid-20th century when Alan Turing proposed the Turing Test as a measure of machine intelligence, leading to early efforts in building simple symbolic AI systems; however, significant progress was slowed by the limits of computing power and understanding of AI's fundamental constructs until the development of the first successful AI programs like ELIZA (1966) and SHRDLU (1972).
- Progress accelerated with

In [10]:
# TODO Part B: Chain-of-Thought comparison (5 points)
#
# Design TWO prompts:
# 1. A math/reasoning problem WITHOUT "think step by step"
# 2. The SAME problem WITH "think step by step" in the system message
#
# Send both to GPT-4o and Phi-4-mini-instruct.
# Observe: does CoT help the smaller model get the right answer?
#
# Hints:
# - Use the same client.chat.completions.create pattern
# - Compare the outputs: does the smaller model benefit more from CoT?
# - Write 2-3 sentences of analysis at the end
models_to_test = ["gpt-4o", "Phi-4-mini-instruct"]
cot_prompts = [
    {
        "name": "Without CoT",
        "system": "You are a helpful math assistant.",
        "user": "Roger has 5 tennis balls. He buys 2 more cans of tennis balls. Each can has 3 tennis balls. How many tennis balls does he have now?"
    },
    {
        "name": "With CoT (zero-shot)",
        "system": "You are a helpful math assistant. Think step by step before giving your final answer.",
        "user": "Roger has 5 tennis balls. He buys 2 more cans of tennis balls. Each can has 3 tennis balls. How many tennis balls does he have now?"
    },
]

# TODO: Send each prompt to both models, print results
for i, prompt in enumerate(cot_prompts):
    print(f"\nCOT TEST {i+1}: {prompt['name']}")
    for model_name in models_to_test:
        try:
            response = client.chat.completions.create(
                model=model_name,
                messages=[
                    {"role": "system", "content": prompt["system"]},
                    {"role": "user", "content": prompt["user"]}
                ]
            )

            print(f"\nModel: {model_name}")
            print(response.choices[0].message.content)

        except Exception as e:
            print(f"\nModel: {model_name}")
            print(f"ERROR: {e}")

# TODO: Write your analysis (2-3 sentences) as a print statement

print("Analysis: In this case, we can see that both with and without CoT, gpt-4o and Phi-4 mini-instruct both get the correct answer, which is 11 tennis balls. In fact, the overall structure of both answers is nearly identical with and without CoT gpt-4o outlines the 'thinking' process with step-by-step process, and phi-4-mini uses a brief but methodical response.")
pass


COT TEST 1: Without CoT

Model: gpt-4o
Let's solve the problem step by step:

1. **Roger starts with 5 tennis balls**.
2. **He buys 2 cans of tennis balls**, and each can contains 3 balls.
   - The total number of balls in the cans is \( 2 \times 3 = 6 \).

3. **Add the 6 balls from the cans to the 5 balls Roger already has**:
   \[
   5 + 6 = 11
   \]

**Roger now has 11 tennis balls.**

Model: Phi-4-mini-instruct
Roger initially has 5 tennis balls. He buys 2 more cans, and each can has 3 tennis balls. So, the number of tennis balls he gets from the cans is 2 cans * 3 balls/can = 6 balls.

Adding the new balls to his initial amount, Roger now has 5 + 6 = 11 tennis balls.

COT TEST 2: With CoT (zero-shot)

Model: gpt-4o
Let's solve this step by step:

1. **Start with the tennis balls Roger already has**: Roger initially has **5 tennis balls**.

2. **Calculate the number of tennis balls in the cans he buys**: Each can contains **3 tennis balls**, and Roger buys **2 cans**.  
   
   So,

## Problem 3: Observing the Effect of Supervised Fine-Tuning (20 points)

### Background

Supervised Fine-Tuning (SFT) is the first step of post-training that transforms a base language model (which just does next-token prediction) into an instruction-following assistant.

### Recap from lecture

- **Base model behavior:** Given "How to fix a car?", it completes with "How to fix a bike?" (pattern continuation, not an answer).
- **After SFT:** Given "How to fix a car?", it generates a helpful, structured answer.
- SFT training data = (instruction, response) pairs.

### What you will do

Instead of training a model from scratch (which is slow), you will use the Azure API to **directly observe the difference** between base-model-style behavior and post-trained behavior. You will:

1. Simulate "base model" behavior by prompting GPT-4o to act as a text completer (no instruction following).
2. Observe normal instruction-following (the result of SFT + RLHF).
3. Design your own SFT training examples for a specific task.

In [11]:
# This cell uses the same Azure OpenAI client from Problem 2.
# Make sure you have already run the client setup cell above.

In [12]:
# TODO Part A: Simulating "Base Model" vs "Instruct Model" behavior (2 points)
#
# You will prompt GPT-4o in two ways to observe what SFT does:
# 1. Tell the model to act as a PURE TEXT COMPLETER (not helpful, just continues text)
# 2. Tell the model to act as a HELPFUL ASSISTANT (normal behavior after SFT)
#
# Hints:
# - For "base model" simulation, use a system message like:
#   "You are a text completion engine. You do NOT answer questions. You simply
#    continue the text as if autocompleting a document. Do not be helpful."
# - For "instruct" mode, use: "You are a helpful assistant. Answer clearly."
# - Use the same user prompt for both and compare the outputs.

prompts_to_compare = [
    "Explain the moon landing to a 6 year old in a few sentences.",
    "Write a Python function that adds two numbers.",
    "What is the capital of France?",
]

base_model_msg = "You are a text completion engine. You do NOT answer questions. You simply continue the text as if autocompleting a document. Do not be helpful."
instruct_msg = "You are a helpful assistant. Answer clearly"

for user_prompt in prompts_to_compare:
    print(f"\nPrompt: \"{user_prompt}\"")
    print("-" * 60)

    # TODO: Send prompt with "base model" system message
    base_response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {
                "role": "system",
                "content": base_model_msg
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ]
    )
    print(f"  [Base model simulation]: {base_response.choices[0].message.content[:200]}")

    # TODO: Send prompt with "instruct" system message
    instruct_response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {
                "role": "system",
                "content": instruct_msg
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ]
    )
    print(f"  [After SFT (instruct)]: {instruct_response.choices[0].message.content[:200]}")
    pass


Prompt: "Explain the moon landing to a 6 year old in a few sentences."
------------------------------------------------------------
  [Base model simulation]: A long time ago, some brave astronauts took a trip to the moon in a big rocket ship. They landed a special spaceship on the moon, walked around wearing funny space suits, and even brought some moon ro
  [After SFT (instruct)]: A long time ago, some very smart people built a big rocket called Apollo 11 to visit the Moon, which is like Earth's tiny neighbor in space. Three astronauts flew in the rocket, and two of them landed

Prompt: "Write a Python function that adds two numbers."
------------------------------------------------------------
  [Base model simulation]: ```python
def add_numbers(a, b):
    return a + b
```
  [After SFT (instruct)]: Certainly! Here is a simple Python function that adds two numbers:

```python
def add_numbers(a, b):
    """
    Adds two numbers and returns the result.

    Parameters:
    a (int or f

In [16]:
# TODO Part B: Design SFT Training Data (3 points)
#
# Imagine you are building a customer service chatbot for a PIZZA SHOP.
# Create 5 training examples that teach the model to:
# 1. Answer questions about the menu
# 2. Take orders politely
# 3. Handle complaints gracefully
#
# Format: list of dicts with "instruction" and "response" keys.
# Then test your examples by using them as few-shot prompts.
#
# Hints:
# - Think about what customers commonly ask at a pizza shop
# - Responses should be polite, helpful, and concise
# - After defining examples, construct a messages list with the first 3 as few-shot
# - Send a NEW question and see if the model follows the pattern

sft_training_examples = [
    {
    "instruction": "My pizza arrived cold.",
    "response": "I'm sorry to hear that. We'd be happy to remake your pizza or provide a refund. Which would you prefer?"
    },
    {
    "instruction":"How large is the medium pizza?",
    "response":"The medium pizza is 14 inches in diameter",
    },
    {
    "instruction":"What is the most popular pizza?",
    "response":"Our most popular pizza is pepperoni",
    },
    {
    "instruction": "I'd like to order a large cheese pizza.",
    "response": "Absolutely! I've added one large cheese pizza to your order. Would you like any drinks or sides with that?"
    },
    {
    "instruction":"What toppings come on the meat lover's pizza",
    "response":"The meat lover's pizza comes with sausage, pepperoni, ham, cheese, and tomato sauce.",
    } 
]

print("Your SFT Training Data:")
for i, example in enumerate(sft_training_examples):
    print(f"  Example {i+1}: {example['instruction']} -> {example['response'][:50]}...")

# TODO: Build few-shot messages from your examples and test on a new question
few_shot_messages = [{"role": "system", "content": "You are a friendly pizza shop assistant."}]
for ex in sft_training_examples[:3]:
    few_shot_messages.append({"role": "user", "content": ex["instruction"]})
    few_shot_messages.append({"role": "assistant", "content": ex["response"]})
few_shot_messages.append({"role": "user", "content": "I'd like to order a small hawaiian pizza"})
response = client.chat.completions.create(model="gpt-4o", messages=few_shot_messages, temperature=0.0)
print(f"Model response: {response.choices[0].message.content}")

Your SFT Training Data:
  Example 1: My pizza arrived cold. -> I'm sorry to hear that. We'd be happy to remake yo...
  Example 2: How large is the medium pizza? -> The medium pizza is 14 inches in diameter...
  Example 3: What is the most popular pizza? -> Our most popular pizza is pepperoni...
  Example 4: I'd like to order a large cheese pizza. -> Absolutely! I've added one large cheese pizza to y...
  Example 5: What toppings come on the meat lover's pizza -> The meat lover's pizza comes with sausage, peppero...
Model response: Great choice! A small Hawaiian pizza typically comes with ham, pineapple, and cheese. Would you like to customize it in any way, or should I place the order as is?


In [15]:
# TODO Part C: Reflection (answer in print statements)
#
# Based on your observations in Parts A and B, answer:
# 1. How does the "base model simulation" output differ from the "instruct" output?
# 2. Why does SFT need (instruction, response) PAIRS rather than just raw text?
# 3. What would happen if your SFT training data had poor quality responses?

# TODO: Write your answers
print("1. The responses in the 'base' model situation are less informational than the 'instruct' situation. For example, in the moon landing one, there are more details such as the name of the rocket in the 'instruct' prompt response. The responses are also more thorough/detailed. For example, the python function in the 'instruct' response has a comment annotating the function.")
print("2. Supervised fine tuning needs instruction response pairs so that the the 'true'/expected response can be compared to what the model actually produces for a given prompt. The delta between the two is used to compute loss, which is then backpropagated to update the model weights.")
print("3. If my SFT training data had poor quality responses, then the model weights would probably be update in the wrong direction/ the model quality would degrade. This would cause it to give worse responses in practice.")

1. The responses in the 'base' model situation are less informational than the 'instruct' situation. For example, in the moon landing one, there are more details such as the name of the rocket in the 'instruct' prompt response. The responses are also more thorough/detailed. For example, the python function in the 'instruct' response has a comment annotating the function.
2. Supervised fine tuning needs instruction response pairs so that the the 'true'/expected response can be compared to what the model actually produces for a given prompt. The delta between the two is used to compute loss, which is then backpropagated to update the model weights.
3. If my SFT training data had poor quality responses, then the model weights would probably be update in the wrong direction/ the model quality would degrade. This would cause it to give worse responses in practice.
